RetailPulse 360

Notebook 04 — Review & Sentiment Intelligence (Shoe Reviews)

Goal: Rossmann, our store network, and H&M covered "when," "where," and "what" — none of
them tell us how customers actually feel about the shoes they bought. This notebook uses
real Amazon footwear review data to extract sentiment per product/category, and specifically
mine sizing-related complaints ("runs small," "true to size," etc.) to validate or correct
the size-curve assumptions we'll build in Notebook 05.

Input: Men_Women_Shoes_Reviews (Kaggle)
Output: review_sentiment.csv, sizing_feedback.csv

In [1]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# 2. LOAD SHOE REVIEWS
# ============================================================

REVIEWS_PATH = "/kaggle/input/datasets/hamaz911/men-women-shoes-reviews/Shoes_Data.csv"

reviews = pd.read_csv(REVIEWS_PATH)

print("Reviews loaded successfully.")
print("Shape:", reviews.shape)
print("\nColumns:", list(reviews.columns))
reviews.head()

Reviews loaded successfully.
Shape: (1230, 8)

Columns: ['title', 'price', 'rating', 'total_reviews', 'product_description', 'reviews', 'reviews_rating', 'Shoe Type']


,title,price,rating,total_reviews,product_description,reviews,reviews_rating,Shoe Type
0,CLYMB Outdoor Sports Running Shoes for Mens Boy,₹279.00,2.9 out of 5 stars,2389 ratings,Elevate your style with this classy pair of Ru...,Not happy with product|| It's not as expected....,1.0 out of 5 stars|| 1.0 out of 5 stars|| 3.0 ...,Men
1,Bourge Men's Loire-z126 Running Shoes,₹479.00,3.9 out of 5 stars,11520 ratings,The product will be an excellent pick for you....,Memory cushioning in these shoes is the best f...,5.0 out of 5 stars|| 1.0 out of 5 stars|| 5.0 ...,Men
2,T-Rock Men's Sneaker,₹430.00,3.3 out of 5 stars,1251 ratings,Flaunt with these stylish and unique red casua...,Worth to its amount|| Go for it|| Perfect|| 5 ...,5.0 out of 5 stars|| 5.0 out of 5 stars|| 5.0 ...,Men
3,Robbie jones Sneakers Casual Canvas Fabric Col...,₹499.00,4.2 out of 5 stars,3 ratings,Robbie Jones Shoes Are Designed To Keeping In ...,Sup quality|| Good but not expected|| Awesome 👌.!,5.0 out of 5 stars|| 3.0 out of 5 stars|| 5.0 ...,Men
4,Sparx Men's Sd0323g Sneakers,₹499.00,4.2 out of 5 stars,20110 ratings,Sparx is a spectacular range of footwear from ...,Best|| Satisfied!|| Affordable beauty 😘😘😘😘 the...,5.0 out of 5 stars|| 5.0 out of 5 stars|| 5.0 ...,Men


In [3]:
# 3. DATA QUALITY — VALIDATE RAW REVIEWS DATA
# ============================================================

print("Total products:", len(reviews))

print("\nDuplicate titles:", reviews["title"].duplicated().sum())

print("\nMissing values:")
missing = reviews.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nShoe Type values:")
print(reviews["Shoe Type"].value_counts())

# CRITICAL CHECK: does the split count of 'reviews' match 'reviews_rating' per row?
review_counts = reviews["reviews"].fillna("").apply(lambda x: len(x.split("||")))
rating_counts = reviews["reviews_rating"].fillna("").apply(lambda x: len(x.split("||")))

mismatch = (review_counts != rating_counts).sum()
print(f"\nRows where review count != rating count: {mismatch} out of {len(reviews)}")
print(f"Total individual reviews across all products (if aligned): {review_counts.sum()}")

Total products: 1230

Duplicate titles: 328

Missing values:
None

Shoe Type values:
Shoe Type
Men      856
Women    374
Name: count, dtype: int64

Rows where review count != rating count: 0 out of 1230
Total individual reviews across all products (if aligned): 9958


In [4]:
# 3b. INVESTIGATE — WHY ARE THERE DUPLICATE TITLES?
# ============================================================

dupe_titles = reviews[reviews["title"].duplicated(keep=False)].sort_values("title")
print("Rows involved in duplicate titles:", len(dupe_titles))

# check a few examples: are these truly identical rows, or same title with different data?
sample_dupe_title = dupe_titles["title"].iloc[0]
print(f"\nExample — all rows with title: '{sample_dupe_title}'")
print(dupe_titles[dupe_titles["title"] == sample_dupe_title][["title","price","rating","total_reviews","Shoe Type"]])

# are they FULLY identical rows (all columns), or just same title with different content?
fully_identical = reviews.duplicated().sum()
print("\nFully identical rows (every column matches):", fully_identical)
print("Same-title-but-different-content rows:", len(dupe_titles) - fully_identical if fully_identical <= len(dupe_titles) else "check manually")

Rows involved in duplicate titles: 510

Example — all rows with title: 'AADI Men's Running Shoe'
                       title    price              rating total_reviews  \
41   AADI Men's Running Shoe  ₹349.00  3.5 out of 5 stars   155 ratings   
116  AADI Men's Running Shoe  ₹399.00  3.6 out of 5 stars   182 ratings   

    Shoe Type  
41        Men  
116       Men  

Fully identical rows (every column matches): 215
Same-title-but-different-content rows: 295


In [5]:
# 3c. REMOVE TRUE DUPLICATES ONLY
# ============================================================
# Drop the 215 fully-identical rows. Keep the 295 same-title-different-
# content rows — these are distinct listings with genuinely different
# review data, not duplicates.

reviews = reviews.drop_duplicates().reset_index(drop=True)

print("Rows after removing true duplicates:", len(reviews))
assert len(reviews) == 1230 - 215, "Unexpected row count after dedup"

Rows after removing true duplicates: 1015


In [6]:
# 4. PARSE NUMERIC FIELDS FROM TEXT
# ============================================================
# price, rating, and total_reviews are stored as formatted text
# ("₹279.00", "2.9 out of 5 stars", "2389 ratings") — extract the
# actual numbers.

reviews["price_inr"] = (
    reviews["price"].str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

reviews["avg_rating"] = (
    reviews["rating"].str.extract(r"([\d.]+)").astype(float)
)

reviews["total_reviews_count"] = (
    reviews["total_reviews"].str.extract(r"([\d,]+)")[0]
    .str.replace(",", "", regex=False)
    .astype(float)
)

print("Parsed numeric fields:")
print(reviews[["price_inr", "avg_rating", "total_reviews_count"]].describe())

Parsed numeric fields:
         price_inr   avg_rating  total_reviews_count
count  1015.000000  1015.000000          1015.000000
mean   1789.546788     3.941379           783.252217
std    1564.919370     0.426353          3034.054929
min     127.000000     1.000000             1.000000
25%     569.000000     3.800000            27.000000
50%    1399.000000     4.000000           123.000000
75%    2403.500000     4.200000           455.500000
max    7992.000000     5.000000         42193.000000


In [8]:
# 5. EXPLODE INTO ONE ROW PER INDIVIDUAL REVIEW
# ============================================================

reviews["review_list"] = reviews["reviews"].str.split(r"\|\|")
reviews["rating_list"] = reviews["reviews_rating"].str.split(r"\|\|")

# sanity check before exploding: list lengths must still match per row
len_mismatch = (
    reviews["review_list"].apply(len) != reviews["rating_list"].apply(len)
).sum()
print("Rows with mismatched list lengths after split:", len_mismatch)
assert len_mismatch == 0, "Review/rating lists don't align — investigate before exploding"

# explode both columns together, keeping pairs aligned
reviews_long = reviews.explode(["review_list", "rating_list"]).reset_index(drop=True)

# clean up: strip whitespace, parse rating_list text -> numeric
reviews_long["review_text"] = reviews_long["review_list"].str.strip()
reviews_long["individual_rating"] = (
    reviews_long["rating_list"].str.extract(r"([\d.]+)").astype(float)
)

print("Total individual reviews:", len(reviews_long))
assert len(reviews_long) == 8208, "Row count doesn't match expected total after dedup"

reviews_long[["title", "Shoe Type", "review_text", "individual_rating"]].head(10)

Rows with mismatched list lengths after split: 0
Total individual reviews: 8208


,title,Shoe Type,review_text,individual_rating
0,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Not happy with product,1.0
1,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,It's not as expected.,1.0
2,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,AVERAGE PRODUCT,3.0
3,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Pic more beautiful,3.0
4,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Got damage product. But quality is average for...,3.0
5,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Bad product different from what was listed,2.0
6,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Worst product,1.0
7,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Don't buy,2.0
8,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Low quality makes pain on heals by sharp edges...,1.0
9,CLYMB Outdoor Sports Running Shoes for Mens Boy,Men,Do not buy it anyway,1.0


In [9]:
# 6. SENTIMENT SCORING (VADER — pretrained, not trained by us)
# ============================================================
import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_compound_sentiment(text):
    if pd.isna(text) or text.strip() == "":
        return np.nan
    return sia.polarity_scores(text)["compound"]

reviews_long["sentiment_score"] = reviews_long["review_text"].apply(get_compound_sentiment)

print("Sentiment score distribution:")
print(reviews_long["sentiment_score"].describe())

# quick sanity check: does sentiment correlate with the star rating?
print("\nAverage sentiment score by individual star rating:")
print(reviews_long.groupby("individual_rating")["sentiment_score"].mean())

Sentiment score distribution:
count    8208.000000
mean        0.179744
std         0.400646
min        -0.888500
25%         0.000000
50%         0.226300
75%         0.492700
max         0.960100
Name: sentiment_score, dtype: float64

Average sentiment score by individual star rating:
individual_rating
1.0   -0.232760
2.0   -0.109770
3.0    0.070337
4.0    0.293854
5.0    0.351691
Name: sentiment_score, dtype: float64


In [10]:
# 7. EXTRACT SIZING FEEDBACK VIA KEYWORD PATTERNS
# ============================================================

def classify_sizing(text):
    if pd.isna(text):
        return None
    t = text.lower()
    if any(p in t for p in ["runs small", "too small", "size small", "smaller than", "tight fit", "narrow fit"]):
        return "runs_small"
    if any(p in t for p in ["runs large", "runs big", "too big", "too large", "loose fit", "bigger than"]):
        return "runs_large"
    if any(p in t for p in ["true to size", "perfect fit", "fits perfectly", "fits well", "correct size"]):
        return "true_to_size"
    return None

reviews_long["sizing_feedback"] = reviews_long["review_text"].apply(classify_sizing)

sizing_counts = reviews_long["sizing_feedback"].value_counts(dropna=False)
print("Sizing feedback breakdown:")
print(sizing_counts)
print(f"\n% of reviews with explicit sizing mention: {reviews_long['sizing_feedback'].notna().mean()*100:.1f}%")

Sizing feedback breakdown:
sizing_feedback
None            8074
true_to_size      70
runs_small        43
runs_large        21
Name: count, dtype: int64

% of reviews with explicit sizing mention: 1.6%


In [11]:
# 8. SIZING FEEDBACK BY GENDER — IS THERE ENOUGH TO BE DIRECTIONAL?
# ============================================================

sizing_by_gender = (
    reviews_long[reviews_long["sizing_feedback"].notna()]
    .groupby(["Shoe Type", "sizing_feedback"])
    .size()
    .unstack(fill_value=0)
)
print(sizing_by_gender)
print("\nTotal sizing mentions per gender:")
print(reviews_long[reviews_long["sizing_feedback"].notna()]["Shoe Type"].value_counts())

sizing_feedback  runs_large  runs_small  true_to_size
Shoe Type                                            
Men                      15          33            53
Women                     6          10            17

Total sizing mentions per gender:
Shoe Type
Men      101
Women     33
Name: count, dtype: int64


In [12]:
# 9. AGGREGATE SENTIMENT BY GENDER
# ============================================================
# Note: this dataset only splits by Men/Women (Shoe Type) — no
# Casual/Formal distinction like H&M had. We aggregate at the
# granularity the real data actually supports, not a finer one we'd
# have to fabricate.

sentiment_by_gender = (
    reviews_long.groupby("Shoe Type")["sentiment_score"]
    .agg(["mean", "median", "std", "count"])
    .rename(columns={"mean": "avg_sentiment", "median": "median_sentiment", "count": "review_count"})
)
print(sentiment_by_gender)

           avg_sentiment  median_sentiment       std  review_count
Shoe Type                                                         
Men             0.182499           0.22630  0.398109          6624
Women           0.168222           0.13785  0.411013          1584


In [13]:
# 10. SAVE OUTPUTS
# ============================================================

sentiment_by_gender.to_csv("review_sentiment.csv")
print("Saved review_sentiment.csv")

sizing_summary = (
    reviews_long[reviews_long["sizing_feedback"].notna()]
    .groupby(["Shoe Type", "sizing_feedback"])
    .size()
    .reset_index(name="mention_count")
)
sizing_summary["confidence"] = "low"  # documented: only 1.6% of reviews had explicit sizing mentions
sizing_summary["usage_note"] = "Directional signal only — not sufficient volume to drive precise size-curve corrections. See Notebook 04 documentation."

sizing_summary.to_csv("sizing_feedback.csv", index=False)
print("Saved sizing_feedback.csv")
print(sizing_summary)

Saved review_sentiment.csv
Saved sizing_feedback.csv
  Shoe Type sizing_feedback  mention_count confidence  \
0       Men      runs_large             15        low   
1       Men      runs_small             33        low   
2       Men    true_to_size             53        low   
3     Women      runs_large              6        low   
4     Women      runs_small             10        low   
5     Women    true_to_size             17        low   

                                          usage_note  
0  Directional signal only — not sufficient volum...  
1  Directional signal only — not sufficient volum...  
2  Directional signal only — not sufficient volum...  
3  Directional signal only — not sufficient volum...  
4  Directional signal only — not sufficient volum...  
5  Directional signal only — not sufficient volum...  


In [14]:
# 11. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 04 SUMMARY")
print(f"Products analyzed: {len(reviews)}")
print(f"Individual reviews extracted: {len(reviews_long)}")
print(f"Sentiment scoring: VADER (pretrained), validated against star ratings (monotonic)")
print(f"Sizing mentions: {reviews_long['sizing_feedback'].notna().sum()} ({reviews_long['sizing_feedback'].notna().mean()*100:.1f}% of reviews) — LOW confidence, directional only")
print(f"Documented gap: no Kids category in this dataset — will use industry-standard assumptions")
print(f"Output files: review_sentiment.csv, sizing_feedback.csv")
print("\n✓ Notebook 04 completed successfully.")

NOTEBOOK 04 SUMMARY
Products analyzed: 1015
Individual reviews extracted: 8208
Sentiment scoring: VADER (pretrained), validated against star ratings (monotonic)
Sizing mentions: 134 (1.6% of reviews) — LOW confidence, directional only
Documented gap: no Kids category in this dataset — will use industry-standard assumptions
Output files: review_sentiment.csv, sizing_feedback.csv

✓ Notebook 04 completed successfully.


In [15]:
# 12. ZIP ARTIFACTS FOR LOCAL DOWNLOAD
# ============================================================
import shutil
from pathlib import Path

OUTPUT_FILES = [Path("review_sentiment.csv"), Path("sizing_feedback.csv")]
ZIP_NAME = "notebook_04_reviews_sentiment_artifact"
ARTIFACT_DIR = Path("notebook_04_outputs")

missing = [f for f in OUTPUT_FILES if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing output file(s): {missing}. Run the save cells first."
    )

ARTIFACT_DIR.mkdir(exist_ok=True)
for f in OUTPUT_FILES:
    shutil.copy(f, ARTIFACT_DIR / f.name)

zip_path = shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir=ARTIFACT_DIR.name)

print("ZIP created successfully:")
print(zip_path)
print(f"ZIP size: {Path(zip_path).stat().st_size / 1024:.2f} KB")
print(f"Contents: {[f.name for f in OUTPUT_FILES]}")

ZIP created successfully:
/kaggle/working/notebook_04_reviews_sentiment_artifact.zip
ZIP size: 0.77 KB
Contents: ['review_sentiment.csv', 'sizing_feedback.csv']
